In [1]:
from datetime import datetime

print(f"Timestamp: {datetime.now()}")

Timestamp: 2025-12-10 09:33:27.375322


# Recluster neurons

**Pinned Environment:** [`envs/sc-scvi.yaml`](../.../envs/sc-scvi.yaml)  

In [2]:
from pathlib import Path
import os
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import sparse
import warnings
import session_info
import sys

In [3]:
plt.rcParams['figure.figsize'] = (3, 3)
plt.rcParams['figure.dpi'] = 150

### Setup

In [4]:
sys.path.append(str(Path.cwd().resolve().parents[2]))

from config.paths import BASE_DIR

input_dir = BASE_DIR / "data/h5ad/export_04/04a_neurons"
adata_path = input_dir / "scanvi_neuron_adata.h5ad"
bdata_path = input_dir / "seurat_neuron_adata.h5ad"

output_dir = BASE_DIR / "data/h5ad/export_04/04b_reclustered"
output_dir.mkdir(parents=True, exist_ok=True)

In [5]:
adata = sc.read_h5ad(adata_path)
bdata = sc.read_h5ad(bdata_path)

In [6]:
print(adata)
print("-"*40

SyntaxError: incomplete input (456995090.py, line 2)

# Neighbors graph

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI_scanvi_neuron', random_state = 0)
sc.pp.neighbors(bdata, use_rep = 'X_scVI_seurat_neuron', random_state = 0)

In [ ]:
sc.tl.umap(adata, random_state = 0)
sc.tl.umap(bdata, random_state = 0)

In [ ]:
sc.tl.leiden(adata, key_added='neuron_recluster', resolution=0.5)
sc.tl.leiden(bdata, key_added='neuron_recluster', resolution=0.5)

In [ ]:
def assign_cell_type_colors(adata, key="cell_type"):
    """
    Assigns a tab10 color palette to a categorical obs field.
    If there are more than 10 categories, the palette cycles.
    """
    # Ensure categorical
    adata.obs[key] = adata.obs[key].astype("category")
    
    cats = adata.obs[key].cat.categories
    n = len(cats)

    # tab10 gives 10 colors; using modulo lets us cycle if n > 1=20
    base_palette = sns.color_palette("tab20", 20)

    palette = [mcolors.to_hex(base_palette[i % 20]) for i in range(n)]

    # assign colors in category order
    adata.uns[f"{key}_colors"] = palette

In [ ]:
assign_cell_type_colors(adata,  key="scanvi_labels")
assign_cell_type_colors(adata,  key="seurat_labels")
assign_cell_type_colors(bdata,  key="scanvi_labels")
assign_cell_type_colors(bdata,  key="seurat_labels")

In [ ]:
sc.pl.umap(adata, color = 'scanvi_labels', frameon = False, title = 'scANVI labels')
sc.pl.umap(bdata, color = 'seurat_labels', frameon = False, title = 'Seurat labels')
sc.pl.umap(adata, color = 'neuron_recluster', frameon = False, title = 'Neuron cluster, scANVI')
sc.pl.umap(bdata, color = 'neuron_recluster', frameon = False, title = 'Neuron cluster, Seurat')

# Summary

In [ ]:
# Seurat summary as that's what we will likely proceed with
egfp = bdata.obs["EGFP_Seq1_hi"].astype(bool)
dtom = bdata.obs["dTomato_Seq2_hi"].astype(bool)
dual = bdata.obs["EGFP_dTomato_dual_hi"].astype(bool)

egfp_single = egfp & ~dtom
dtom_single = dtom & ~egfp
dual_pos    = dual

# Build a small table
df = pd.DataFrame({
    "barcode": bdata.obs.index,
    "seurat_label": bdata.obs["seurat_labels"],
    "scanvi_label": bdata.obs["scanvi_labels"],
    "confidence": bdata.obs["confidence"],
    "EGFP_single": egfp_single.values,
    "dTom_single": dtom_single.values,
    "Dual_positive": dual_pos.values,
})

In [ ]:
df[df["EGFP_single"]][["barcode", "seurat_label", "scanvi_label", "confidence"]]

In [ ]:
df[df["dTom_single"]][["barcode", "seurat_label", "scanvi_label", "confidence"]]

In [ ]:
df[df["Dual_positive"]][["barcode", "seurat_label", "scanvi_label", "confidence"]]

# Export

In [ ]:
adata_path = os.path.join(output_dir, 'neurons-scanvi.h5ad')
bdata_path = os.path.join(output_dir, 'neurons-seurat.h5ad')

adata.write_h5ad(adata_path, compression='gzip') # didnt run 12/9 evening
adata.write_h5ad(bdata_path, compression='gzip')

print(adata_path)
print(bdata_path)

## Session info

In [ ]:
print('active IDE: sc-spatial-gpu')
print('active conda environment:', os.path.basename(sys.prefix))
session_info.show()

In [ ]:
adata

# test recompute neighbors use scanvi

In [ ]:
sc.pl.umap(bdata, color = ['neuron_recluster', 'seurat_labels'], frameon = False, title = 'scANVI labels')

In [ ]:
bdata.obs.EGFP_dTomato_dual_hi.value_counts()

In [ ]:
bdata_subset = bdata[bdata.obs['total_counts'] > 1000].copy()
sc.pl.umap(bdata_subset, color = 'seurat_labels', frameon = False, title = 'scANVI labels')

In [ ]:
bdata_subset.obs.EGFP_dTomato_dual_hi.value_counts()

# test

In [ ]:
bdata_copy = bdata.copy()

In [ ]:
bdata.X = bdata.layers['log1p'].copy()

In [ ]:
sc.pp.regress_out(bdata, ["total_counts"])
sc.pp.scale(bdata, max_value=10)

In [ ]:
sc.pp.pca(bdata, n_comps = 10)
sc.pp.neighbors(bdata,random_state = 0, n_neighbors = 10)
#sc.pp.neighbors(bdata,random_state = 0, use_rep = 'X_scVI_seurat_neuron', n_neighbors = 30)


In [ ]:
sc.tl.umap(bdata, random_state = 0)

In [ ]:
sc.pl.umap(bdata, color = 'seurat_labels', frameon = False, title = 'scANVI labels')

In [ ]:
import matplotlib.pyplot as plt

# Column to visualize
col = "EGFP_dTomato_dual_hi"

# Get UMAP coordinates
x, y = bdata.obsm["X_umap"][:, 0], bdata.obsm["X_umap"][:, 1]

# Boolean mask
mask = bdata.obs[col].astype(bool).values

plt.figure(figsize=(3, 3))

# Plot FALSE (background)
plt.scatter(
    x[~mask],
    y[~mask],
    s=5,
    c="lightgray",
    linewidths=0,
    alpha=0.4,
)

# Plot TRUE (highlight)
plt.scatter(
    x[mask],
    y[mask],
    s=25,
    c="gold",         # ← change color if you want
    edgecolors="black",
    linewidths=0.6,
    alpha=1.0,
)

# Minimalist formatting
ax = plt.gca()
ax.set_aspect("equal", "box")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(bdata, color = ['total_counts', 'seurat_labels', 'neuron_recluster'], frameon = False, title = 'scANVI labels', legend_loc = 'on data')

In [ ]:
import pandas as pd

# Boolean mask for dual-positive cells
dual_mask = bdata.obs["EGFP_dTomato_dual_hi"].astype(bool)

# Build a small dataframe for only dual-positive rows
df_dual = pd.DataFrame({
    "barcode": bdata.obs.index[dual_mask],
    "seurat_label": bdata.obs["seurat_labels"][dual_mask],
    "scanvi_label": bdata.obs["scanvi_labels"][dual_mask],
    "neuron_recluster": bdata.obs["neuron_recluster"][dual_mask],
    "total_counts": bdata.obs["total_counts"][dual_mask],
})

df_dual

In [ ]:
import scanpy as sc
import pandas as pd

cluster_col = "neuron_recluster"
target_cluster = "8"   # ← change this if you want a different cluster

# make sure categorical
adata.obs[cluster_col] = adata.obs[cluster_col].astype("category")

print("Cluster counts:")
print(adata.obs[cluster_col].value_counts(), "\n")

# ------------------------------
# Run DE: cluster 8 vs the rest
# ------------------------------
sc.tl.rank_genes_groups(
    adata,
    groupby=cluster_col,
    groups=[target_cluster],
    reference="rest",     # compares cluster 8 vs all other clusters
    method="wilcoxon"
)

# ------------------------------
# Extract results
# ------------------------------
markers_c8 = sc.get.rank_genes_groups_df(adata, group=target_cluster)

print(f"Top DE genes: cluster {target_cluster} vs ALL other clusters\n")
display(markers_c8.head(30))

# ------------------------------
# Plot top markers
# ------------------------------
sc.pl.rank_genes_groups(adata, groups=[target_cluster], n_genes=20, sharey=False)